# Closira multi-task DistilBERT (intent + emotion)
Runtime → Change runtime type → **T4 GPU**. Builds the datasets (streams raw data, ~25 min for TWCS), trains, evaluates, and zips the model for `backend/ml/artifacts/multitask/`.

In [ ]:
!git clone -q https://github.com/tanushreenerella/closira-it-helpdesk.git closira
%cd closira
!pip install -q "transformers>=4.40" scikit-learn pandas joblib

In [ ]:
# Skip this cell if you uploaded backend/ml/data/processed/*.csv.gz instead
!python -m backend.ml.data.build_datasets

In [ ]:
!python -m backend.ml.baselines

In [ ]:
!python -m backend.ml.multitask --epochs 6 --batch-size 32 --fp16

In [ ]:
!python -m backend.ml.evaluate_ood --model all

In [ ]:
!cd backend/ml && zip -qr /content/multitask_model.zip artifacts/multitask results
from google.colab import files
files.download('/content/multitask_model.zip')

Unzip into `backend/ml/` locally; the backend picks up `artifacts/multitask/` automatically on the next start.

## Ablation: 1 shared model vs 2 single-task models
Run the setup and data cells above (1–2), then these. All three runs use identical splits, seed, hyperparameters and early stopping; each head gets the same number of updates per epoch. About 30 min on a T4. Writes `results/ablation_{both,intent,emotion}.json` — `artifacts/multitask` is not touched.

In [ ]:
# shared model, retrained in this session so all three are comparable
!python -m backend.ml.multitask --tasks both --epochs 6 --batch-size 32 --fp16 --out backend/ml/artifacts/ablation_both --results-name ablation_both

In [ ]:
!python -m backend.ml.multitask --tasks intent --epochs 6 --batch-size 32 --fp16

In [ ]:
!python -m backend.ml.multitask --tasks emotion --epochs 6 --batch-size 32 --fp16

In [ ]:
# accuracy, macro-F1, frustrated F1, parameters, CPU latency per message (1 thread)
!python -m backend.ml.ablation

In [ ]:
!cd backend/ml && zip -qr /content/ablation_results.zip results/ablation_*.json
from google.colab import files
files.download('/content/ablation_results.zip')

Unzip into `backend/ml/` locally (results only — the ablation models stay in Colab).